**Inference Optimization Lab**

1.  Define & Invoke
2.  Metrics
3.  Compare Models
4.  Tool Use
5.  Prompt Caching
6.  Agent Metrics

**Key Metrics**

TTFT — Time to First Token (perceived speed)
TTC — Time to Completion (total request duration)
OTPS — Output Tokens Per Second (throughput after streaming starts)
Cost — $/1M tokens (input vs output)


# **Part 0: Setup**


Import modules and define API key.


In [ ]:
%%capture
!pip3 install anthropic --upgrade --break-system-packages
!pip3 install claude-agent-sdk --break-system-packages
!pip3 install tabulate python-dotenv --break-system-packages

import importlib
import tabulate
from dotenv import load_dotenv
import os

api_key = "sk-ant-..."  # <-- Paste your API key here

Test Claude Invokation


In [5]:
import anthropic

# Initialize the Anthropic client
client = anthropic.Anthropic(api_key=api_key)

# Models we'll benchmark
MODEL_SONNET = "claude-sonnet-4-6"
MODEL_HAIKU = "claude-haiku-4-5-20251001"
MODEL_OPUS = "claude-opus-4-8"

# Default model for exercises
DEFAULT_MODEL = MODEL_SONNET

# Health check: make a simple Messages API call
try:
    response = client.messages.create(model=DEFAULT_MODEL, max_tokens=5, messages=[
                                      {"role": "user", "content": "Ping"}])
    print(f"Health check passed: {response.content[0].text}")
    print(f"Using model: {DEFAULT_MODEL}")
except anthropic.APIError as e:
    print(f"Health check failed: {e}")
    raise

Health check passed: Pong! 
Using model: claude-sonnet-4-6


Define data classes (You don't need to modify these, just understand what they are)


In [6]:
from dataclasses import dataclass, field
from typing import List, Optional
from tabulate import tabulate
import statistics


@dataclass
class BenchmarkResult:
    """Timing, tokens, and cost for a single API call."""
    ttft: float                    # Time to First Token (seconds)
    total_time: float              # Time to Completion / TTC (seconds)
    tokens_per_second: float       # Legacy: output_tokens / ttc
    input_tokens: int
    output_tokens: int
    endpoint: str
    model: str
    test_name: str
    otps: Optional[float] = None   # Output Tokens Per Second
    cost: Optional[float] = None   # Cost in dollars
    cache_creation_input_tokens: Optional[int] = None
    cache_read_input_tokens: Optional[int] = None


@dataclass
class BenchmarkSuite:
    """Collects results across multiple runs."""
    results: List[BenchmarkResult] = field(default_factory=list)

    def add_result(self, result: BenchmarkResult):
        self.results.append(result)

    def clear(self):
        self.results = []

    def summary(self, group_by: str = "test_name") -> str:
        if not self.results:
            return "No results."

        groups = {}
        for r in self.results:
            key = getattr(r, group_by)
            if key not in groups:
                groups[key] = []
            groups[key].append(r)

        rows = []
        for name, group in groups.items():
            ttfts = [r.ttft * 1000 for r in group]
            ttcs = [r.total_time * 1000 for r in group]
            throughputs = [r.otps or r.tokens_per_second for r in group]
            costs = [r.cost for r in group if r.cost is not None]

            row = [
                name,
                len(group),
                f"{statistics.mean(ttfts):.0f}",
                f"{statistics.mean(ttcs):.0f}",
                f"{statistics.mean(throughputs):.1f}",
            ]
            if costs:
                row.append(f"${sum(costs)*1000:.4f}")  # Cost per 1000 calls
            rows.append(row)

        headers = ["Test", "Runs", "TTFT(ms)", "TTC(ms)", "OTPS"]
        if any(r.cost for r in self.results):
            headers.append("$/1K calls")
        return tabulate(rows, headers=headers, tablefmt="grid")


suite = BenchmarkSuite()
print("✓ BenchmarkSuite ready (with cost tracking)")

✓ BenchmarkSuite ready (with cost tracking)


# **Part 1: Metrics**


Every measurement starts with the same streaming pattern: open a stream, watch for events, capture timing. Let's extract this into a reusable helper so we don't repeat it.

We will also add a calculation for TTFT.

_TTFT (Time To First Token)_ = time elapsed from sending the request to receiving the first token back from the model.


In [7]:
import time


def _stream_request(prompt, model=DEFAULT_MODEL, max_tokens=256):
    """Low-level helper: stream a request, return raw timing + response."""
    ttft = None
    start_time = time.perf_counter()
    with client.messages.stream(
        model=model, max_tokens=max_tokens, messages=[
            {"role": "user", "content": prompt}]
    ) as stream:
        # Loop through events — when you see "content_block_start", record TTFT.
        for event in stream:
            if ttft is None and event.type == "content_block_start":
                ttft = time.perf_counter() - start_time
        # After the loop, call stream.get_final_message() to get the response.
        response = stream.get_final_message()

    total_time = time.perf_counter() - start_time
    return ttft, total_time, response


print("✓ _stream_request helper ready")

✓ _stream_request helper ready


Let's display the TTFT result


In [8]:
ttft, total_time, response = _stream_request(
    "What is 2 + 2? Answer in one word.")

ttft_ms = ttft * 1000
ttc_ms = total_time * 1000

print(f"Response: {response.content[0].text}")
print(f"TTFT: {ttft_ms:.0f}ms")
print(f"TTC:  {ttc_ms:.0f}ms")

Response: **Four**
TTFT: 987ms
TTC:  1099ms


Now let's compute throughput from the raw timing data.

**Formulas:**

- **OTPS** (Output Tokens Per Second) = `output_tokens / (ttc - ttft)`

Why subtract TTFT? OTPS measures generation speed _after streaming starts_. The time before the first token is waiting, not generating.


In [9]:
def compute_otps(ttft, total_time, output_tokens):
    # OTPS measures generation speed *after* the first token arrives.
    gen_time = total_time - ttft
    if gen_time <= 0:
        return 0.0, gen_time
    otps = output_tokens / gen_time
    return otps, gen_time


ttft, total_time, response = _stream_request(
    "What is 2 + 2? Answer in one word.")

tokens = response.usage.output_tokens
otps, gen_time = compute_otps(ttft, total_time, tokens)

print(f"Response: {response.content[0].text}")
print(f"TTFT: {ttft * 1000:.0f}ms")
print(f"TTC:  {total_time * 1000:.0f}ms")
print(f"OTPS: {otps:.1f} tokens/sec ({tokens} tokens / {gen_time:.3f}s)")

Response: **Four**
TTFT: 1000ms
TTC:  1078ms
OTPS: 76.9 tokens/sec (6 tokens / 0.078s)


Every API call has a cost based on token usage. Input and output tokens are priced differently. Lets track cost of our API calls.


In [10]:
PRICING = {
    "claude-sonnet-4-6": {"input": 3.00, "output": 15.00},
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
    "claude-opus-4-8": {"input": 5.00, "output": 25.00},
}


def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> tuple[float, float, float]:
    prices = PRICING.get(model, {"input": 3.00, "output": 15.00})
    # Prices are per 1M tokens.
    input_cost = input_tokens / 1_000_000 * prices["input"]
    output_cost = output_tokens / 1_000_000 * prices["output"]
    total = input_cost + output_cost
    return input_cost, output_cost, total


ttft, total_time, response = _stream_request(
    "What is 2 + 2? Answer in one word.")

usage = response.usage
input_cost, output_cost, cost = calculate_cost(
    DEFAULT_MODEL, usage.input_tokens, usage.output_tokens)

print(f"Response: {response.content[0].text}")
print(f"Tokens: {usage.input_tokens} in / {usage.output_tokens} out")
print(
    f"Cost:   ${cost:.6f} (input: ${input_cost:.6f}, output: ${output_cost:.6f})")

Response: **Four**
Tokens: 21 in / 6 out
Cost:   $0.000153 (input: $0.000063, output: $0.000090)


Now we compose `_stream_request`, `compute_otps`, and `calculate_cost` into a single `measure_streaming_latency` function. No new logic — just calling the functions we already built.


In [11]:
def measure_streaming_latency(
    prompt: str,
    model: str = DEFAULT_MODEL,
    max_tokens: int = 256,
    test_name: str = "streaming"
) -> BenchmarkResult:
    ttft, total_time, response = _stream_request(prompt, model, max_tokens)
    usage = response.usage
    _, _, cost = calculate_cost(model, usage.input_tokens, usage.output_tokens)
    otps, _ = compute_otps(ttft, total_time, usage.output_tokens)

    return BenchmarkResult(
        ttft=ttft,
        total_time=total_time,
        tokens_per_second=otps,
        input_tokens=usage.input_tokens,
        output_tokens=usage.output_tokens,
        endpoint="1p",
        model=model,
        test_name=test_name,
        otps=otps,
        cost=cost,
    )

# **Part 2: Compare Models**


Run the same prompt through Haiku, Sonnet, and Opus to compare their performance profiles.


In [12]:
def percentile(data, p):
    sorted_data = sorted(data)
    idx = (len(sorted_data) - 1) * p / 100
    low = int(idx)
    high = min(low + 1, len(sorted_data) - 1)
    fraction = idx - low
    return sorted_data[low] + fraction * (sorted_data[high] - sorted_data[low])


suite.clear()
PROMPT = "What is machine learning? Answer in 2 sentences."

models = [
    (MODEL_HAIKU, "haiku"),
    (MODEL_SONNET, "sonnet"),
    (MODEL_OPUS, "opus"),
]

for model_id, model_name in models:
    print(f"\nBenchmarking {model_name}...")

    for i in range(5):
        result = measure_streaming_latency(
            PROMPT, model=model_id, test_name=model_name)
        suite.add_result(result)

        ttft_ms = result.ttft * 1000
        ttc_ms = result.total_time * 1000

        print(
            f"  Run {i+1}: TTFT={ttft_ms:.0f}ms, TTC={ttc_ms:.0f}ms, OTPS={result.otps:.1f}, Cost=${result.cost:.6f}")

print("\n" + suite.summary())


Benchmarking haiku...
  Run 1: TTFT=741ms, TTC=1152ms, OTPS=128.9, Cost=$0.000284
  Run 2: TTFT=796ms, TTC=1132ms, OTPS=161.1, Cost=$0.000289
  Run 3: TTFT=789ms, TTC=1068ms, OTPS=186.8, Cost=$0.000279
  Run 4: TTFT=857ms, TTC=1327ms, OTPS=104.1, Cost=$0.000264
  Run 5: TTFT=715ms, TTC=1034ms, OTPS=162.9, Cost=$0.000279

Benchmarking sonnet...
  Run 1: TTFT=943ms, TTC=1968ms, OTPS=49.8, Cost=$0.000822
  Run 2: TTFT=1039ms, TTC=1538ms, OTPS=96.2, Cost=$0.000777
  Run 3: TTFT=990ms, TTC=2122ms, OTPS=50.4, Cost=$0.000912
  Run 4: TTFT=1159ms, TTC=1769ms, OTPS=83.6, Cost=$0.000822
  Run 5: TTFT=1054ms, TTC=1999ms, OTPS=55.0, Cost=$0.000837

Benchmarking opus...
  Run 1: TTFT=1276ms, TTC=2507ms, OTPS=69.1, Cost=$0.002245
  Run 2: TTFT=817ms, TTC=1792ms, OTPS=91.3, Cost=$0.002345
  Run 3: TTFT=919ms, TTC=2100ms, OTPS=81.3, Cost=$0.002520
  Run 4: TTFT=900ms, TTC=1619ms, OTPS=122.3, Cost=$0.002320
  Run 5: TTFT=938ms, TTC=1877ms, OTPS=90.5, Cost=$0.002245

+--------+--------+------------+---

# **Part 3: Introduce Tool Calling**


Tool use adds a round-trip: Request → tool_use → Execute → Result → Response. First, define a calculator tool with a JSON Schema for its inputs.


In [13]:
import json

CALCULATOR_TOOL = {
    "name": "calculator",
    "description": "Performs basic arithmetic operations.",
    "input_schema": {
        "type": "object",
        "properties": {
            "operation": {
                "type": "string",
                "enum": ["add", "subtract", "multiply", "divide"],
                "description": "The arithmetic operation to perform.",
            },
            "operands": {
                "type": "array",
                "items": {"type": "number"},
                "description": "The two numbers to operate on.",
            },
        },
        "required": ["operation", "operands"]
    }
}

print("Calculator tool:")
print(json.dumps(CALCULATOR_TOOL, indent=2))

Calculator tool:
{
  "name": "calculator",
  "description": "Performs basic arithmetic operations.",
  "input_schema": {
    "type": "object",
    "properties": {
      "operation": {
        "type": "string",
        "enum": [
          "add",
          "subtract",
          "multiply",
          "divide"
        ],
        "description": "The arithmetic operation to perform."
      },
      "operands": {
        "type": "array",
        "items": {
          "type": "number"
        },
        "description": "The two numbers to operate on."
      }
    },
    "required": [
      "operation",
      "operands"
    ]
  }
}


This function executes the calculator tool locally. **No changes needed.**


In [14]:
def execute_calculator(operation: str, operands: list) -> float:
    """Execute a calculator operation."""
    a, b = operands[0], operands[1]

    if operation == "add":
        return a + b
    elif operation == "subtract":
        return a - b
    elif operation == "multiply":
        return a * b
    elif operation == "divide":
        return a / b
    else:
        raise ValueError(f"Unknown: {operation}")


print(f"Test: 42 * 17 = {execute_calculator('multiply', [42, 17])}")

Test: 42 * 17 = 714


The function handles the full round-trip:

1. Send request with tool → 2. Parse tool_use → 3. Execute locally → 4. Send result → 5. Get response


In [15]:
def measure_tool_use_latency(prompt: str, model: str = DEFAULT_MODEL, max_tokens: int = 256):
    """Measure full round-trip latency for a tool use request."""

    start_time = time.perf_counter()

    # First API call with the calculator tool available.
    first_response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        tools=[CALCULATOR_TOOL],
        messages=[{"role": "user", "content": prompt}],
    )

    ttft = time.perf_counter() - start_time

    # Find the tool_use block in the first response.
    tool_use_block = None
    for block in first_response.content:
        if block.type == "tool_use":
            tool_use_block = block
            break

    if tool_use_block is None:
        # No tool use - return early
        total_time = time.perf_counter() - start_time
        return ttft, total_time, "No tool used", first_response.usage.input_tokens, first_response.usage.output_tokens

    # Execute the tool locally, then send the result back in a second API call.
    result = execute_calculator(
        tool_use_block.input["operation"], tool_use_block.input["operands"])

    second_response = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        tools=[CALCULATOR_TOOL],
        messages=[
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": first_response.content},
            {
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_use_block.id,
                        "content": str(result),
                    }
                ],
            },
        ],
    )

    total_time = time.perf_counter() - start_time

    # Extract final text
    final_text = ""
    for block in second_response.content:
        if hasattr(block, "text"):
            final_text += block.text

    total_input = first_response.usage.input_tokens + \
        second_response.usage.input_tokens
    total_output = first_response.usage.output_tokens + \
        second_response.usage.output_tokens

    return ttft, total_time, final_text, total_input, total_output

In [16]:
# Test tool use
ttft, total, text, in_tok, out_tok = measure_tool_use_latency(
    "What is 42 * 17? Use the calculator.")

print(f"TTFT: {ttft*1000:.0f}ms")
print("\n✓ Tool use working!")

TTFT: 1801ms

✓ Tool use working!


Let's measure the latency overhead of tool use.


In [17]:
# Compare: With Tool vs Without Tool
suite.clear()

print("Without tool:")
for i in range(5):
    result = measure_streaming_latency(
        "What is forty-two times seventeen? Show your work.",
        test_name="no_tool"
    )
    suite.add_result(result)
    print(f"  Run {i+1}: TTFT={result.ttft*1000:.0f}ms, TTC={result.total_time*1000:.0f}ms, Cost=${result.cost:.6f}")

print("\nWith tool:")
for i in range(5):
    ttft, total_time, text, in_tok, out_tok = measure_tool_use_latency(
        "What is 42 * 17? Use the calculator."
    )
    _, _, cost = calculate_cost(DEFAULT_MODEL, in_tok, out_tok)
    otps, gen_time = compute_otps(ttft, total_time, out_tok)

    result = BenchmarkResult(
        ttft=ttft,
        total_time=total_time,
        tokens_per_second=otps,
        input_tokens=in_tok,
        output_tokens=out_tok,
        endpoint="1p",
        model=DEFAULT_MODEL,
        test_name="with_tool",
        otps=otps,
        cost=cost,
    )
    suite.add_result(result)
    print(
        f"  Run {i+1}: TTFT={ttft*1000:.0f}ms, TTC={total_time*1000:.0f}ms, Cost=${cost:.6f}")

print("\n" + suite.summary())

Without tool:
  Run 1: TTFT=906ms, TTC=3057ms, Cost=$0.002352
  Run 2: TTFT=1023ms, TTC=2663ms, Cost=$0.001632
  Run 3: TTFT=1023ms, TTC=3071ms, Cost=$0.002352
  Run 4: TTFT=1024ms, TTC=2635ms, Cost=$0.001632
  Run 5: TTFT=958ms, TTC=3030ms, Cost=$0.002322

With tool:
  Run 1: TTFT=1963ms, TTC=3348ms, Cost=$0.005691
  Run 2: TTFT=2218ms, TTC=3584ms, Cost=$0.005691
  Run 3: TTFT=2047ms, TTC=3488ms, Cost=$0.005691
  Run 4: TTFT=1887ms, TTC=3381ms, Cost=$0.005691
  Run 5: TTFT=2048ms, TTC=3568ms, Cost=$0.005691

+-----------+--------+------------+-----------+--------+--------------+
| Test      |   Runs |   TTFT(ms) |   TTC(ms) |   OTPS | $/1K calls   |
+===========+========+============+===========+========+==============+
| no_tool   |      5 |        987 |      2891 |   69.6 | $10.2900     |
+-----------+--------+------------+-----------+--------+--------------+
| with_tool |      5 |       2033 |      3473 |   75.1 | $28.4550     |
+-----------+--------+------------+-----------+------

# **Part 4: Prompt Caching**

Cache processed prompts so the API skips re-reading them on subsequent calls.

**How it works:**

1. Add `cache_control: {"type": "ephemeral"}` to content blocks
2. First call **writes** to cache (costs 25% more on cached tokens)
3. Subsequent calls **read** from cache (costs 90% less on cached tokens)

**Requirements:**

- Minimum 1,024 tokens in the cached block
- Cache lives for 5 minutes (ephemeral)
- Cached content must be an exact prefix match


In [18]:
# Build a system prompt large enough for caching (>1024 tokens)
SYSTEM_PROMPT = """You are an expert API documentation assistant.
You help developers understand REST API design, authentication patterns,
security best practices, rate limiting, pagination, error handling,
versioning strategies, webhook design, and performance optimization.
Always provide concrete examples with HTTP methods and status codes.
""" * 20

# A system block with caching enabled on the large prompt.
system_block = [
    {
        "type": "text",
        "text": SYSTEM_PROMPT,
        "cache_control": {"type": "ephemeral"},
    }
]


def cached_request(question):
    start = time.perf_counter()
    response = client.messages.create(
        model=DEFAULT_MODEL,
        max_tokens=100,
        system=system_block,
        messages=[{"role": "user", "content": question}],
    )
    elapsed = time.perf_counter() - start
    return response, elapsed


# Call 1: Cold — creates the cache
r1, time1 = cached_request("What is REST?")
print(f"Cold call: {time1 * 1000:.0f}ms")
print(f"  Cache created: {r1.usage.cache_creation_input_tokens or 0} tokens")
print(f"  Cache read:    {r1.usage.cache_read_input_tokens or 0} tokens")
print(f"  Input tokens:  {r1.usage.input_tokens}")

# Call 2: Warm — reads from cache (different question, same system prompt)
r2, time2 = cached_request("What is OAuth?")
print(f"\nWarm call: {time2 * 1000:.0f}ms")
print(f"  Cache created: {r2.usage.cache_creation_input_tokens or 0} tokens")
print(f"  Cache read:    {r2.usage.cache_read_input_tokens or 0} tokens")
print(f"  Input tokens:  {r2.usage.input_tokens}")

Cold call: 3082ms
  Cache created: 1162 tokens
  Cache read:    0 tokens
  Input tokens:  9

Warm call: 3372ms
  Cache created: 0 tokens
  Cache read:    1162 tokens
  Input tokens:  9


As conversations grow, the API re-reads every prior message on each turn. Cache breakpoints fix this — we cache the system prompt and mark the latest assistant reply, so subsequent turns only process the new question. Watch `cached` grow as the conversation builds up.


In [19]:
SYSTEM_PROMPT = """You are a helpful API design consultant. You specialize in REST API design,
authentication patterns, rate limiting, pagination, error handling, versioning strategies,
webhook design, and performance optimization. Always provide concrete examples with HTTP
methods, status codes, request/response schemas, and curl commands.
""" * 20

SYSTEM = [{"type": "text", "text": SYSTEM_PROMPT,
           "cache_control": {"type": "ephemeral"}}]

# Multi-turn caching: mark the PREVIOUS assistant message with cache_control
# BEFORE calling the API — this tells Claude "cache everything up to here".


def chat(messages, new_question):
    # Clear old cache breakpoints (convert list content back to plain text)
    for msg in messages:
        if msg["role"] == "assistant" and isinstance(msg["content"], list):
            msg["content"] = msg["content"][0]["text"]

    # Add cache_control to the last assistant message (if any) so the API
    # caches the whole conversation prefix up to and including that turn.
    if messages and messages[-1]["role"] == "assistant":
        last = messages[-1]
        last["content"] = [{
            "type": "text",
            "text": last["content"],
            "cache_control": {"type": "ephemeral"},
        }]

    messages.append({"role": "user", "content": new_question})

    start = time.perf_counter()
    response = client.messages.create(
        model=MODEL_SONNET,
        max_tokens=300,
        system=SYSTEM,
        messages=messages,
    )
    elapsed = time.perf_counter() - start

    answer = response.content[0].text
    messages.append({"role": "assistant", "content": answer})

    return answer, elapsed, response.usage


conversation = []
questions = [
    "Design a REST API for a todo app. Include all endpoints.",
    "Now add authentication. What changes?",
    "Add rate limiting. How should the headers look?",
    "Now add team support — users can share todo lists.",
    "Summarize the full API design so far.",
]

for i, question in enumerate(questions):
    answer, elapsed, usage = chat(conversation, question)

    cached = usage.cache_read_input_tokens or 0
    created = usage.cache_creation_input_tokens or 0

    print(
        f"Turn {i+1}: {elapsed * 1000:.0f}ms | cached: {cached} | created: {created}")

Turn 1: 5357ms | cached: 0 | created: 1242
Turn 2: 4763ms | cached: 1242 | created: 318
Turn 3: 5369ms | cached: 1560 | created: 313
Turn 4: 5693ms | cached: 1873 | created: 316
Turn 5: 5592ms | cached: 2189 | created: 317
